# v1401 冻结双模型 fast 提交

上传本 Notebook，以及同目录 `v1002-model.json`、`v1101-model.json`，共三个 UTF-8 文本文件。
两个 JSON 均为原组件的完整精度权重、配置、统计量和训练／推理源码，不含离线预测答案。

组件为 v1002 第 4 轮 EMA 与 v1101 第 5 轮 EMA；组合规则在本次评估前固定，不新增训练、不搜索权重。
每个预测时点，分别把两套完整股票池预测按总体标准差标准化，再等权平均；常数组件映射为零。
factor 是排序分数，不是校准收益。全部股票计算后才组合，不在小批量内计算截面标准化。

`main(datasources, start_date, end_date)` 使用 `bar1m`、`bar5m`、`bar30m` 的逻辑行情源。
共享一次行情读取、归一化和切窗；两个冻结网络分别按股票日共享历史及当天因果前缀。
网络总参数 4,715,286，网络推理成本高于单个组件；没有测量平台运行时间，不承诺平台速度倍数。
结束日期字符串包含全天，结束时间戳包含该分钟。导入不查询行情，推理不训练、不读取标签、不访问外网。

依赖及完整训练代码在各 JSON 的 requirements、sources、training_sources 中；推理还需要平台 dai。
显式调用 `reproduce("reproduced-v1401", "/path/to/data", training_confirmed=True)` 会串行复现两个组件，
返回各自运行目录；目标目录必须不存在。重新训练受硬件与数值环境影响，不保证恢复同一权重。
复现后的两套完整预测可交给 `combine_predictions(left, right)`，使用同一固定组合函数。
通常仅调用 main 使用本次冻结权重即可。下方调试单元格不启动训练。


In [ ]:
"""三文件冻结组合入口；源码和配置由导出器填入，不在导入时查询行情。"""

_CONFIG = {'solution_id': 'v1401', 'seed': 42, 'parent_solution_id': 'v1002', 'parent_run_id': 'v1002-current-temporal-experts-s42', 'model': {'family': 'hierarchical_gru', 'composition': 'frozen_cross_section_standardized_equal_mean', 'components': [{'solution_id': 'v1002', 'run_id': 'v1002-current-temporal-experts-s42', 'weight': 0.5, 'model_sha256': '5f61413f1651eac718a239da255659e62628a5ab6d455705bf86a84fcc4c29eb', 'config_sha256': '663c86e8f461914111984188c016525837cb8dd09ebbdf8ddd8ced443a099832', 'selected_epoch': 4, 'weight_source': 'ema', 'prediction_sha256': {'validation': '3766245165116f40caefdc086c033e11f47993eee41fad572f744ee7562cd7db', 'test': '8b821de1d89a6f53cfbd02affb8b6a36dc2c2b966e812d0260472e2c6f1c4c43'}}, {'solution_id': 'v1101', 'run_id': 'v1101-structured-fields-s42', 'weight': 0.5, 'model_sha256': 'fae2ed405cc228d50bb39b4ea6f06a08d5b23978fb7bda677cdb72770db46dde', 'config_sha256': 'b900489bf28c08619f5e6b043e4dd8a96aef26d7c8506f78236a5930132eaecf', 'selected_epoch': 5, 'weight_source': 'ema', 'prediction_sha256': {'validation': '9d1ea661f3b7d8791add21c4d727d675dc7ac895d5bc1c0f5a58c38731ce157e', 'test': '0219108cd7eed2981e3d82a6d4941fbb3c6215bf8cf1d2c2942305f4f0609223'}}]}, 'training': {'mode': 'none_frozen_components', 'epochs': 0, 'cpu_threads': 4}, 'inference': {'stock_day_batch_size': 128, 'precision': 'float32_network_float64_aggregation'}, 'references': {'local_best': 'v1002-current-temporal-experts-s42', 'parent': 'v1002-current-temporal-experts-s42', 'platform_best': None}, 'aggregation': {'component_weights': [0.5, 0.5], 'centering': 'complete_timestamp_cross_section', 'std_ddof': 0, 'constant_component': 'zero', 'label_usage': False, 'weight_search': False}, 'declared_at': '2026-10-22T11:29:30+08:00', 'research_basis': 'docs/frozen-ensemble-research-20261022.md'}
_OWN_SOURCES = {'solutions/v1401/v1401_combine.py': {'text': '"""完整预测时点内标准化后等权组合；不读取标签。"""\n\nimport numpy as np\nimport pandas as pd\n\nKEYS = [\'date\', \'instrument\']\n\n\ndef combine_scores(keys, scores):\n    if list(keys.columns) != KEYS or keys[KEYS].isna().any().any() or keys.duplicated(KEYS).any():\n        raise ValueError(\'组合要求唯一、非空的 date/instrument 键\')\n    scores = np.asarray(scores, dtype=np.float64)\n    if scores.shape != (len(keys), 2) or not np.isfinite(scores).all():\n        raise ValueError(\'组合要求两列有限、完整对齐的分数\')\n    result = keys.copy().reset_index(drop=True)\n    if not len(result):\n        result[\'factor\'] = np.array([], dtype=np.float64)\n        return result\n    values = pd.DataFrame(scores)\n    centered = values - values.groupby(result.date).transform(\'mean\')\n    scale = centered.pow(2).groupby(result.date).transform(\'mean\').pow(.5)\n    normalized = centered.div(scale.where(scale > 0)).fillna(0)\n    result[\'factor\'] = normalized.mean(axis=1).to_numpy(dtype=np.float64)\n    if not np.isfinite(result.factor).all():\n        raise FloatingPointError(\'组合输出非有限\')\n    return result\n\n\ndef align_components(left, right):\n    for frame in (left, right):\n        if list(frame.columns) != KEYS + [\'factor\'] or frame.duplicated(KEYS).any():\n            raise ValueError(\'组件预测格式或键不合法\')\n    aligned = left.merge(right, on=KEYS, how=\'outer\', suffixes=(\'_left\', \'_right\'), validate=\'one_to_one\')\n    scores = aligned[[\'factor_left\', \'factor_right\']].to_numpy(dtype=np.float64)\n    if not np.isfinite(scores).all():\n        raise ValueError(\'两个组件预测必须完整、有限且键相同\')\n    return aligned[KEYS], scores\n', 'sha256': '325509f731b256604331608a105f3a744aa2b7a78c18e76efa7b9c4a9c9c55a1'}, 'solutions/v1401/v1401_model.py': {'text': '"""两个已有冻结 GRU 的组件分数；截面组合在全部股票计算后执行。"""\n\nimport torch\nfrom torch import nn\n\nfrom v1002_model import MultiFrequencyGRU as TemporalModel\nfrom v1101_model import MultiFrequencyGRU as StructuredModel\n\n\nclass FrozenEnsemble(nn.Module):\n    def __init__(self, configs):\n        super().__init__()\n        if [config[\'solution_id\'] for config in configs] != [\'v1002\', \'v1101\']:\n            raise ValueError(\'冻结组合必须按 v1002、v1101 顺序加载\')\n        if configs[0][\'data\'] != configs[1][\'data\']:\n            raise ValueError(\'共享输入要求完全相同的数据契约\')\n        self.components = nn.ModuleList([TemporalModel(configs[0]), StructuredModel(configs[1])])\n        self.requires_grad_(False)\n\n    def component_scores(self, windows, clock):\n        return torch.stack([model(windows, clock) for model in self.components], dim=-1)\n\n    def component_scores_group(self, windows, clock):\n        return torch.stack([model.forward_group(windows, clock)[..., 0] for model in self.components], dim=-1)\n', 'sha256': '3e6852bed60e99c8a5c7e2ff003f54dd9eff5d9e8989ef2f6921d7aab1ba5b8e'}, 'solutions/v1401/v1401_fast.py': {'text': '"""冻结双模型共享股票日取窗，整日全部股票计算后组合。"""\r\n\r\nimport torch\r\nimport numpy as np\r\n\r\nfrom v1002_data import prediction_clock_values\r\nfrom v1401_combine import combine_scores, KEYS\r\n\r\n\r\ndef predict_all_clocks(model, windows, latest_clock):\r\n    if model.training:\r\n        raise ValueError("shared inference requires eval mode")\r\n    return model.component_scores_group(windows, latest_clock)\r\n\r\n\r\ndef predict_day(model, block, device, *, stock_batch_size=128):\r\n    """按股票日复用取窗，保留动态股票池、缺失时点及原键顺序。"""\r\n    if isinstance(stock_batch_size, bool) or not isinstance(stock_batch_size, int) or stock_batch_size < 1:\r\n        raise ValueError("股票日批量必须为正整数")\r\n    if model.training:\r\n        raise ValueError("共享编码仅用于 eval 推理")\r\n    if not len(block.keys):\r\n        return np.empty(0, np.float64)\r\n    if len(np.unique(block.day_indices)) != 1:\r\n        raise ValueError("共享编码入口每次只接收一个交易日")\r\n    _, group = np.unique(block.stock_indices, return_inverse=True)\r\n    count = int(group.max()) + 1\r\n    clock_ids = prediction_clock_values(block.times).argmax(axis=1)\r\n    # 不依赖 keys 原本排序；每只股票选择请求内最晚时点，未来仍由原 batch 路径清零。\r\n    order = np.lexsort((clock_ids, group))\r\n    ends = np.r_[np.flatnonzero(np.diff(group[order])) + 1, len(order)] - 1\r\n    representatives = order[ends]\r\n    values = np.empty((count, 8, 2), np.float64)\r\n    with torch.inference_mode():\r\n        for offset in range(0, count, stock_batch_size):\r\n            selected = representatives[offset:offset + stock_batch_size]\r\n            windows, latest = block.batch(selected)\r\n            tensors = {f: torch.from_numpy(x).to(device) for f, x in windows.items()}\r\n            scores = predict_all_clocks(model, tensors, torch.from_numpy(latest).to(device))\r\n            values[offset:offset + len(selected)] = (\r\n                scores / block.config["inference"]["factor_scale"]).cpu().numpy()\r\n    return combine_scores(block.keys[KEYS], values[group, clock_ids]).factor.to_numpy(dtype=np.float64)\r\n', 'sha256': '61b66ca8707b7d9704135c3a39d4d9110ad1eccd02ae8338597cccf9e4d7cf82'}, 'solutions/v1401/v1401_predict.py': {'text': '"""从平台逻辑行情源推理；复用冻结统计量与因果切窗，不读取标签。"""\r\n\r\nfrom datetime import date, datetime\r\nimport json\r\nfrom pathlib import Path\r\nimport re\r\n\r\nimport torch\r\nimport numpy as np\r\nimport pandas as pd\r\n\r\nfrom v1002_data import WindowBlock, TradingCalendar, regular_times, minute_slots, VALUE_COLUMNS\r\nfrom v1002_schema import TRADING_DAY_WINDOWS\r\nfrom v1401_model import FrozenEnsemble\r\nfrom v1002_text_artifact import decode_state as decode_temporal\r\nfrom v1101_text_artifact import decode_state as decode_structured\r\nfrom v1401_fast import predict_day\r\nfrom v1002_preprocessing import (FIELDS, FREQUENCIES, KEYS, Normalizer, UniverseIndex,\r\n                              population_std, empty_moments, log_values, running_moments)\r\n\r\n\r\ndef local_time(value):\r\n    value = pd.Timestamp(value)\r\n    if pd.isna(value):\r\n        raise ValueError("日期不能为空")\r\n    return value.tz_convert("Asia/Shanghai").tz_localize(None) if value.tzinfo else value\r\n\r\n\r\ndef platform_interval(start_date, end_date):\r\n    start, end = local_time(start_date), local_time(end_date)\r\n    date_only = (isinstance(end_date, str) and re.fullmatch(r"\\d{4}-\\d{2}-\\d{2}", end_date.strip()) is not None\r\n                 or isinstance(end_date, date) and not isinstance(end_date, datetime))\r\n    stop = end.normalize() + pd.Timedelta(days=1) if date_only else end.floor("min") + pd.Timedelta(minutes=1)\r\n    if start >= stop or (not date_only and start > end):\r\n        raise ValueError("非法推理日期区间")\r\n    return start.ceil("min"), stop\r\n\r\n\r\ndef identifier(value):\r\n    if not isinstance(value, str) or not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*", value):\r\n        raise ValueError("非法行情表标识符")\r\n    return value\r\n\r\n\r\ndef read_table(dai_module, table, columns, start, stop, extra=""):\r\n    table = identifier(table)\r\n    columns = [identifier(c) for c in columns]\r\n    query = (f"SELECT {\', \'.join(columns)} FROM {table} WHERE date >= \'{start.isoformat(sep=\' \')}\' "\r\n             f"AND date < \'{stop.isoformat(sep=\' \')}\' {extra}")\r\n    frame = dai_module.query(query, filters={"date": [start.normalize().strftime("%Y-%m-%d"), stop.ceil("D").strftime("%Y-%m-%d")]}).df()\r\n    frame = frame[columns].copy()\r\n    frame["date"] = pd.to_datetime(frame.date)\r\n    if frame[KEYS].isna().any().any() or frame.duplicated(KEYS).any():\r\n        raise ValueError("行情键重复或为空")\r\n    if not frame.empty and not ((frame.date >= start) & (frame.date < stop)).all():\r\n        raise ValueError("行情查询超出区间")\r\n    return frame.sort_values(KEYS).reset_index(drop=True)\r\n\r\n\r\ndef warm_history(dai_module, source, normalizer, start, stop):\r\n    """只为缺少可信自身统计的股票恢复全部已知历史；不计算或依赖未来截面。"""\r\n    if start >= stop:\r\n        return\r\n    complete = {stock for stock, value in normalizer.stocks.items()\r\n                if ((value["count"] >= normalizer.minimum) & (population_std(value) > normalizer.epsilon)).all()}\r\n    observed = dai_module.query(\r\n        f"SELECT DISTINCT instrument FROM {identifier(source)} WHERE date >= \'{start.isoformat(sep=\' \')}\' AND date < \'{stop.isoformat(sep=\' \')}\'",\r\n        filters={"date": [start.normalize().strftime("%Y-%m-%d"), stop.ceil("D").strftime("%Y-%m-%d")]},\r\n    ).df()\r\n    missing = sorted(set(observed.instrument.astype(str)) - complete)\r\n    if not missing:\r\n        return\r\n    quoted = ["\'" + stock.replace("\'", "\'\'") + "\'" for stock in missing]\r\n    extra = "AND instrument IN (" + ",".join(quoted) + ")"\r\n    current = start\r\n    while current < stop:\r\n        end = min((current.to_period("M") + 1).start_time, stop)\r\n        frame = read_table(dai_module, source, KEYS + FIELDS, current, end, extra)\r\n        for stock, part in frame.groupby("instrument", sort=False, observed=True):\r\n            previous = normalizer.history.get(stock)\r\n            if previous is None:\r\n                previous = empty_moments(len(FIELDS)) if normalizer.training else normalizer.stocks.get(stock, empty_moments(len(FIELDS)))\r\n            _, _, _, final = running_moments(log_values(part), previous)\r\n            normalizer.history[stock] = final\r\n        if not frame.empty:\r\n            normalizer.last_time = frame.date.iloc[-1]\r\n        current = end\r\n\r\n\r\ndef prior_trading_days(dai_module, universe_table, start, count):\r\n    """从股票池查询真实交易日期，不用固定自然日或工作日近似。"""\r\n    query = (f"SELECT DISTINCT CAST(date AS DATE) AS date FROM {identifier(universe_table)} "\r\n             f"WHERE date < \'{start.normalize().isoformat(sep=\' \')}\' "\r\n             f"ORDER BY date DESC LIMIT {int(count)}")\r\n    frame = dai_module.query(query).df()\r\n    days = pd.DatetimeIndex(pd.to_datetime(frame.date)).normalize().sort_values().unique()\r\n    if days.hasnans or len(days) > count or (days >= start.normalize()).any():\r\n        raise ValueError("历史交易日查询返回非法日期")\r\n    return days\r\n\r\n\r\ndef predict(datasources, start_date, end_date, *, bundles, ensemble_config, dai_module=None, device=None):\r\n    if dai_module is None:\r\n        import dai as dai_module\r\n    config = bundles[0]["config"]\r\n    preprocessing = bundles[0]["preprocessing"]\r\n    torch.set_num_threads(config["training"]["cpu_threads"])\r\n    torch.backends.cuda.matmul.allow_tf32 = False\r\n    torch.backends.cudnn.allow_tf32 = False\r\n    torch.backends.cudnn.benchmark = False\r\n    device = torch.device(device or ("cuda" if torch.cuda.is_available() else "cpu"))\r\n    model = FrozenEnsemble([bundle["config"] for bundle in bundles]).to(device).eval()\r\n    for component, bundle, decode in zip(model.components, bundles, (decode_temporal, decode_structured)):\r\n        component.load_state_dict(decode(bundle), strict=True)\r\n    start, stop = platform_interval(start_date, end_date)\r\n    sources = {f: identifier(datasources[f]) for f in FREQUENCIES}\r\n    universe_table = "cpt_jyc_2026_instruments"\r\n    previous = prior_trading_days(dai_module, universe_table, start,\r\n                                  max(s["previous_days"] for s in TRADING_DAY_WINDOWS.values()))\r\n    context_start = previous[0] if len(previous) else start.normalize()\r\n    # 只查更早的最后一个股票池快照，确保首个 Bar 的截面不引用未来池。\r\n    prior = dai_module.query(f"SELECT max(date) AS date FROM {universe_table} WHERE date < \'{context_start.isoformat(sep=\' \')}\'").df().date.iloc[0]\r\n    universe_start = min(context_start, pd.Timestamp(prior)) if pd.notna(prior) else context_start\r\n    universe = read_table(dai_module, universe_table, KEYS, universe_start, stop)\r\n    pool_index = UniverseIndex(universe)\r\n    target_keys = universe.loc[(universe.date >= start) & (universe.date < stop), KEYS]\r\n    if target_keys.empty:\r\n        return pd.DataFrame({"date": pd.Series(dtype="datetime64[ns]"), "instrument": pd.Series(dtype="object"),\r\n                             "factor": pd.Series(dtype="float64")})\r\n    context_days = np.sort(universe.loc[universe.date >= context_start, "date"].dt.normalize().unique())\r\n    calendar = TradingCalendar(context_days)\r\n    frequency_starts = {f: calendar.history_start(start, TRADING_DAY_WINDOWS[f]["previous_days"])\r\n                        for f in FREQUENCIES}\r\n    normalizers = {}\r\n    for frequency in FREQUENCIES:\r\n        statistics = bundles[0]["normalization"][frequency]\r\n        fit_start, fit_stop = (pd.Timestamp(statistics["fit_interval"][k]) for k in ("start", "stop"))\r\n        if start < fit_start:\r\n            raise ValueError("请求早于训练数据起点")\r\n        frequency_start = frequency_starts[frequency]\r\n        normalizer = Normalizer(statistics, preprocessing, training=frequency_start < fit_stop)\r\n        warm_start = fit_start if normalizer.training else fit_stop\r\n        warm_history(dai_module, sources[frequency], normalizer, warm_start, frequency_start)\r\n        normalizers[frequency] = normalizer\r\n    stocks = pd.Index(sorted(target_keys.instrument.astype(str).unique()))\r\n    trading_days = set(universe.date.dt.normalize())\r\n    # 动态历史保留源表全部日期／时段；模型网格只由股票池交易日构建。\r\n    days = pd.date_range(context_start, stop - pd.Timedelta(1, "ns"), freq="D")\r\n    tails = {f: (np.array([], np.int64), np.empty((0, len(stocks), 72), np.float32)) for f in FREQUENCIES}\r\n    chunks = []\r\n    with torch.inference_mode():\r\n        for day in days:\r\n            lo = pd.Timestamp(day)\r\n            hi = min(lo + pd.Timedelta(days=1), stop)\r\n            keys = target_keys.loc[(target_keys.date >= lo) & (target_keys.date < hi)].reset_index(drop=True)\r\n            grids = {}\r\n            for frequency in FREQUENCIES:\r\n                if lo < frequency_starts[frequency]:\r\n                    grids[frequency] = tails[frequency]\r\n                    continue\r\n                normalizer = normalizers[frequency]\r\n                if lo >= normalizer.fit_stop:\r\n                    normalizer.training = False\r\n                raw = read_table(dai_module, sources[frequency], KEYS + FIELDS, lo, hi)\r\n                normalized, _ = normalizer.transform(raw, pool_index.eligible(raw))\r\n                times = regular_times([day], frequency) if lo in trading_days else np.array([], np.int64)\r\n                # 即便请求截止盘中，仍保留今天完整槽位；未来数值由切窗显式置零。\r\n                values = np.full((len(times), len(stocks), 72), np.nan, np.float32)\r\n                if not normalized.empty:\r\n                    ti = pd.Index(times).get_indexer(normalized.date.to_numpy(dtype="datetime64[ns]").astype(np.int64))\r\n                    si = stocks.get_indexer(normalized.instrument)\r\n                    valid = (ti >= 0) & (si >= 0)\r\n                    values[ti[valid], si[valid]] = normalized.loc[valid, VALUE_COLUMNS].to_numpy(dtype=np.float32)\r\n                old_times, old_values = tails[frequency]\r\n                grids[frequency] = np.concatenate([old_times, times]), np.concatenate([old_values, values])\r\n                length = TRADING_DAY_WINDOWS[frequency]["previous_days"] * len(minute_slots(frequency))\r\n                tails[frequency] = (grids[frequency][0][-length:].copy(), grids[frequency][1][-length:].copy())\r\n                del raw, normalized\r\n            if len(keys):\r\n                block = WindowBlock(keys, grids, stocks, config, context_days)\r\n                factors = predict_day(model, block, device,\r\n                                      stock_batch_size=ensemble_config["inference"]["stock_day_batch_size"])\r\n                keys["factor"] = factors\r\n                chunks.append(keys)\r\n                del block\r\n    result = pd.concat(chunks, ignore_index=True) if chunks else pd.DataFrame(columns=KEYS + ["factor"])\r\n    result["date"] = pd.to_datetime(result.date)\r\n    result["factor"] = result.factor.astype(np.float64)\r\n    if result.duplicated(KEYS).any() or result[KEYS].isna().any().any() or not np.isfinite(result.factor).all():\r\n        raise ValueError("推理输出键或数值非法")\r\n    if len(result) != len(target_keys):\r\n        raise ValueError("未覆盖完整股票池")\r\n    return result[KEYS + ["factor"]].sort_values(KEYS).reset_index(drop=True)\r\n', 'sha256': '836be618479ab555010432144760dcc5eb29ae2fcb6fb653109be739acc0f2b6'}}
_RUNTIME = None


def _bundle_directory():
    from pathlib import Path
    return Path(__file__).resolve().parent if '__file__' in globals() else Path.cwd()


def _read_bundles():
    import json
    return [json.loads((_bundle_directory() / name).read_bytes().decode('utf-8-sig'))
            for name in ('v1002-model.json', 'v1101-model.json')]


def _load_modules(sources, paths):
    import torch
    import builtins
    from pathlib import Path
    from types import ModuleType
    modules = {}

    def local_import(name, globals=None, locals=None, fromlist=(), level=0):
        if level == 0 and name in modules:
            return modules[name]
        return builtins.__import__(name, globals, locals, fromlist, level)

    for relative in paths:
        module = ModuleType(Path(relative).stem)
        module.__file__ = str(_bundle_directory() / Path(relative).name)
        module.__dict__['__builtins__'] = dict(vars(builtins), __import__=local_import)
        modules[module.__name__] = module
        exec(compile(sources[relative]['text'], module.__file__, 'exec'), module.__dict__)
    return modules


def _runtime():
    global _RUNTIME
    if _RUNTIME is None:
        bundles = _read_bundles()
        sources = {name: entry for bundle in bundles for name, entry in bundle['sources'].items()}
        sources.update(_OWN_SOURCES)
        paths = [name for bundle in bundles for name in bundle['runtime_sources']] + list(_OWN_SOURCES)
        _RUNTIME = bundles, _load_modules(sources, paths)
    return _RUNTIME


def main(datasources, start_date, end_date):
    bundles, modules = _runtime()
    return modules['v1401_predict'].predict(datasources, start_date, end_date,
                                           bundles=bundles, ensemble_config=_CONFIG)


def reproduce(workspace, data_root, *, training_confirmed=False):
    """显式复现两个组件，串行运行；组合函数见 combine_predictions。"""
    from pathlib import Path
    if not training_confirmed:
        raise ValueError('组件训练复现须显式确认')
    workspace = Path(workspace).resolve()
    if workspace.exists():
        raise FileExistsError(workspace)
    workspace.mkdir(parents=True)
    results = {}
    for bundle in _read_bundles():
        version = bundle['config']['solution_id']
        name = version + '_reproduce'
        modules = _load_modules(bundle['sources'], [f'solutions/{version}/{name}.py'])
        results[version] = modules[name].reproduce(bundle, workspace / version, data_root, training_confirmed=True)
    return results


def combine_predictions(v1002_predictions, v1101_predictions):
    """组合两个组件的完整同截面预测，可用于复现后评估。"""
    _, modules = _runtime()
    combine = modules['v1401_combine']
    keys, scores = combine.align_components(v1002_predictions, v1101_predictions)
    return combine.combine_scores(keys, scores)


In [ ]:
if __name__ == "__main__":
    result = main({f: "cpt_jyc_2026_stock_" + f for f in ("bar1m", "bar5m", "bar30m")}, "2024-12-30", "2024-12-31")
    import numpy as np
    import pandas as pd
    assert list(result.columns) == ["date", "instrument", "factor"]
    assert not result.empty and result.factor.dtype == np.dtype("float64")
    assert pd.api.types.is_datetime64_any_dtype(result.date)
    assert result.instrument.map(lambda value: isinstance(value, str)).all()
    assert not result[["date", "instrument"]].isna().any().any()
    assert not result.duplicated(["date", "instrument"]).any()
    assert np.isfinite(result.factor).all()
    print("输出行数：", len(result))
    print(result.dtypes)
    display(result.head())
